In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print(torch.version.cuda)


CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 4070 SUPER
12.1


In [14]:
import h5py

with h5py.File(r'C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\h5_files\FA 47 B1.h5', 'r') as f:
    print(f.keys())               # should show 'features' and 'coords'
    # print(f['coords'].shape)    # (N, D)
    # print(f['coords'][:10])  # first 10 coordinates




<KeysViewHDF5 ['coords', 'features']>


In [4]:
import h5py

with h5py.File(r'C:\Users\Vivian\Documents\CLAM\CLAM\heatmaps\blockmaps\FA 47 B1.h5', 'r') as f:
    print(f.keys())               # should show 'features' and 'coords'
    print(f['coords'].shape)    # (N, D)
    print(f['coords'][:10])  # first 10 coordinates
    print(f['attention_scores'][10:])  # first 10 attention scores


<KeysViewHDF5 ['attention_scores', 'coords']>
(2136, 2)
[[1344  448]
 [1568  448]
 [1120  672]
 [1344  672]
 [1568  672]
 [1792  672]
 [2016  672]
 [ 896  896]
 [1120  896]
 [1344  896]]
[[-5.7015963]
 [-4.9648046]
 [-5.159204 ]
 ...
 [-5.2162457]
 [-3.9081943]
 [-5.5734015]]


In [ ]:
from vis_utils.heatmap_utils import WholeSlideImage
import numpy as np
from PIL import Image
import cv2

def drawHeatmap_blank(scores, coords, patch_size=(224, 224), convert_to_percentiles=True, cmap='jet', alpha=1.0):
    from matplotlib import cm

    # Normalize scores to [0, 1]
    if convert_to_percentiles:
        percentiles = np.array([np.sum(scores <= s) for s in scores]) / len(scores)
    else:
        s_min, s_max = np.min(scores), np.max(scores)
        percentiles = (scores - s_min) / (s_max - s_min + 1e-8)

    # Generate colormap
    colormap = cm.get_cmap(cmap)
    color_mapped = (colormap(percentiles.squeeze())[:, :3] * 255).astype(np.uint8)  # drop alpha

    # Calculate canvas size
    max_x = coords[:, 0].max() + patch_size[0]
    max_y = coords[:, 1].max() + patch_size[1]
    canvas = np.ones((max_y, max_x, 3), dtype=np.uint8) * 255

    for i, (x, y) in enumerate(coords):
        color = color_mapped[i]
        patch = np.ones((patch_size[1], patch_size[0], 3), dtype=np.uint8) * color[np.newaxis, np.newaxis, :]
        patch = (patch * alpha + canvas[y:y+patch_size[1], x:x+patch_size[0]] * (1 - alpha)).astype(np.uint8)
        canvas[y:y+patch_size[1], x:x+patch_size[0]] = patch

    return Image.fromarray(canvas)


In [12]:
import h5py
from vis_utils.heatmap_utils import drawHeatmap
from PIL import Image

h5_path = r'C:\Users\Vivian\Documents\CLAM\CLAM\heatmaps\uni\blockmaps_8\PT 76 B2.h5'
# h5_path = r'C:\Users\Vivian\Documents\CLAM\CLAM\heatmaps\blockmaps_8\PT 76 B2.h5'
slide_path = None  # Set to None to use blank canvas
save_path = r'C:\Users\Vivian\Documents\CLAM\CLAM\heatmaps\uni\test_heatmaps\s8_PT_76 B2_heatmap.png'
# save_path = r'C:\Users\Vivian\Documents\CLAM\CLAM\heatmaps\test_heatmaps_fa_pt\s8_PT_76 B2_heatmap.png'
with h5py.File(h5_path, 'r') as f:
    scores = f['attention_scores'][:]
    coords = f['coords'][:]

heatmap = drawHeatmap_blank(
    scores=scores,
    coords=coords,
    patch_size=(224, 224),
    convert_to_percentiles=True,
    cmap='jet',
    alpha=1.0
)
heatmap.save(save_path)


C:\Users\Vivian\AppData\Local\Temp\ipykernel_41340\1805106329.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap(cmap)
